# National eTenders.gov.in Scraper

Scrapes **https://etenders.gov.in** (NIC eProcurement national portal) for tenders by organisation,
filters by project amount range, exports a styled Excel listing, and downloads BOQ ZIP files.

**Workflow:**
1. Configure organisation names and value range below
2. Run all cells — scraping runs **headless** in the background (no visible browser)
3. For BOQ downloads, Chrome opens **only if CAPTCHA** is needed
4. Output lands in `national_Projects_dep_<timestamp>/<OrgName>/`

In [2]:
!pip install selenium webdriver-manager pandas openpyxl xlrd reportlab

Defaulting to user installation because normal site-packages is not writeable
  Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl.metadata (12 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached wsproto-1.2.0-py3-none-any.whl.metadata (5.6 kB)
  Using cached PySocks-1.7.1-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 2.7 MB/s eta 0:00:0000:0100:01
Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl (27 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 1.9 MB/s eta 0:00:0000:0100:01
Using cached openpyxl-3.1.5-py2.py3-none-any.

In [3]:
import io
import os
import re
import sys
import time
import shutil
import zipfile
import datetime
import logging
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    StaleElementReferenceException,
    ElementClickInterceptedException,
)
from webdriver_manager.chrome import ChromeDriverManager

PORTAL_URL = "https://etenders.gov.in/eprocure/app"
RUPEES_PER_CRORE = 10_000_000

BASE_PATH = os.path.dirname(os.path.abspath("__file__"))

log_dir = os.path.join(BASE_PATH, "logs", "national")
os.makedirs(log_dir, exist_ok=True)
log_ts = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
log_file = os.path.join(log_dir, f"scraper_{log_ts}.log")

logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
console = logging.StreamHandler()
console.setLevel(logging.INFO)
console.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
logging.getLogger().addHandler(console)

logging.info("Logger ready — log file: %s", log_file)
print(f"Log file: {log_file}")

/Users/vakul/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
2026-04-11 13:08:09,933 - INFO - Logger ready — log file: /Users/vakul/Documents/Projects & Ideas/TendersScrapper-RPG/RPGOYAL EPROC Automation/logs/national_scraper_2026-04-11_13-08-09.log


Log file: /Users/vakul/Documents/Projects & Ideas/TendersScrapper-RPG/RPGOYAL EPROC Automation/logs/national_scraper_2026-04-11_13-08-09.log


## Configuration

Set the organisation names (partial / case-insensitive match supported) and tender value range in crore.

In [4]:
# ──────────────────────────────────────────────────
# USER CONFIGURATION — edit these before running
# ──────────────────────────────────────────────────

ORGANISATIONS = [
    "National Highways Authority of India",
    # Add more organisation names here (partial match is fine)
]

MIN_CRORE = 8      # lower bound in crore (inclusive)
MAX_CRORE = 50     # upper bound in crore (inclusive)

DOWNLOAD_BOQS = True   # set False to skip BOQ ZIP downloads

NUM_WORKERS = 4        # parallel headless browsers for scraping tender details

# ──────────────────────────────────────────────────
min_rupees = int(MIN_CRORE * RUPEES_PER_CRORE)
max_rupees = int(MAX_CRORE * RUPEES_PER_CRORE)

date_str = datetime.datetime.now().strftime("%Y-%m-%d")
output_root = os.path.join(BASE_PATH, "results", "national", date_str)
os.makedirs(output_root, exist_ok=True)

print(f"Organisations : {ORGANISATIONS}")
print(f"Value filter  : ₹{MIN_CRORE} Cr – ₹{MAX_CRORE} Cr")
print(f"Output folder : {output_root}")

Organisations : ['National Highways Authority of India']
Value filter  : ₹8 Cr – ₹50 Cr
Output folder : /Users/vakul/Documents/Projects & Ideas/TendersScrapper-RPG/RPGOYAL EPROC Automation/national_Projects_dep_2026-04-11_13-08-49


## Helper Functions

In [5]:
def init_driver(headless=True, download_dir=None):
    """Create a Chrome WebDriver with optional headless mode and download directory."""
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--start-maximized")
    options.page_load_strategy = "eager"
    if download_dir:
        os.makedirs(download_dir, exist_ok=True)
        dl = os.path.abspath(download_dir)
        options.add_experimental_option(
            "prefs",
            {
                "download.default_directory": dl,
                "download.prompt_for_download": False,
                "directory_upgrade": True,
                "safebrowsing.enabled": True,
            },
        )
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)


def sanitize_folder_name(name):
    """Filesystem-safe folder/file name fragment."""
    name = (name or "").strip()
    for c in '<>:"/\\|?*\n\r\t':
        name = name.replace(c, "_")
    name = re.sub(r"_+", "_", name).strip("._ ")
    return name or "department"


def parse_inr_amount_to_rupees(text):
    """Parse tender value strings like '21,50,00,000' into integer rupees."""
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return None
    s = str(text).strip()
    if not s or s.lower() in ("not found", "refer document", "refer tender document"):
        return None
    digits = re.sub(r"[^\d]", "", s)
    if not digits:
        return None
    try:
        return int(digits)
    except ValueError:
        return None


def get_field_text(driver, field_name, timeout=5):
    """Extract a field value from a tender detail table row (NIC layout: bold label | value)."""
    try:
        xpath = f"//td/b[normalize-space()='{field_name}']/ancestor::td/following-sibling::td[1]"
        el = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.XPATH, xpath))
        )
        return el.text.strip()
    except Exception:
        return "Not Found"


def get_zipcode_from_page(driver):
    """Try common NIC labels for PIN / ZIP on the tender detail page."""
    for label in ("Pin Code", "Pincode", "PIN Code", "ZIP Code", "Zipcode", "Postal Code"):
        val = get_field_text(driver, label, timeout=1)
        if val and val != "Not Found":
            return val
    return "Not Found"


def sanitize_project_zip_basename(tender_data, seq_index):
    """Filesystem-safe basename from tender Title or Work Description."""
    raw = (tender_data.get("Title") or "").strip()
    if not raw or raw.lower() == "not found":
        raw = (tender_data.get("Work Description") or "").strip()
    if not raw or raw.lower() == "not found":
        raw = f"tender_{seq_index:04d}"
    base = sanitize_folder_name(raw)
    if len(base) > 150:
        base = base[:150].rstrip("_")
    return base or f"tender_{seq_index:04d}"


def unique_path_in_dir(target_dir, basename, ext=".zip"):
    """Return a unique file path by appending _2, _3, etc. if name exists."""
    p = os.path.join(target_dir, f"{basename}{ext}")
    if not os.path.exists(p):
        return p
    n = 2
    while True:
        p = os.path.join(target_dir, f"{basename}_{n}{ext}")
        if not os.path.exists(p):
            return p
        n += 1


print("Helper functions defined.")

Helper functions defined.


## Navigation — Organisation listing, selection, pagination

In [6]:
def _wait_for_page_table(driver, timeout=15):
    """Wait until a data table is present on the page (common NIC pattern)."""
    try:
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located(
                (By.XPATH, "//table[.//tr[contains(@class,'even') or contains(@class,'odd')]] | //table[@id='table']")
            )
        )
    except TimeoutException:
        pass


def navigate_to_org_listing(driver):
    """Open the portal and navigate to the 'Tender by Organisation' page (headless-safe)."""
    driver.get(PORTAL_URL)
    try:
        link = WebDriverWait(driver, 15).until(
            EC.element_to_be_clickable(
                (By.XPATH, "//a[contains(translate(normalize-space(.), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'tender by organisation')]")
            )
        )
        link.click()
        _wait_for_page_table(driver)
        logging.info("Navigated to Tender by Organisation page.")
    except TimeoutException:
        driver.get(f"{PORTAL_URL}?page=FrontEndTendersByOrganisation&service=page")
        _wait_for_page_table(driver)
        logging.info("Navigated to Tender by Organisation page via direct URL.")


def get_all_organisations(driver):
    """
    Parse the organisation table on the 'Tender by Organisation' page.
    Returns dict: {org_name: click_element_id} where the click element is the
    hyperlinked tender-count number beside each organisation name.
    """
    org_dict = {}
    try:
        table = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "table"))
        )
    except TimeoutException:
        # Fallback: try finding any table with organisation rows
        tables = driver.find_elements(By.TAG_NAME, "table")
        table = None
        for t in tables:
            if len(t.find_elements(By.TAG_NAME, "tr")) > 3:
                table = t
                break
        if table is None:
            logging.error("Could not find organisation table on the page.")
            return org_dict

    rows = table.find_elements(By.TAG_NAME, "tr")
    for row in rows:
        classes = row.get_attribute("class") or ""
        if "list_header" in classes:
            continue
        cells = row.find_elements(By.TAG_NAME, "td")
        if len(cells) >= 3:
            org_name = cells[1].text.strip()
            if not org_name:
                continue
            try:
                anchor = cells[2].find_element(By.TAG_NAME, "a")
                anchor_id = anchor.get_attribute("id")
                if anchor_id:
                    org_dict[org_name] = anchor_id
            except NoSuchElementException:
                pass

    logging.info("Found %d organisations on the listing page.", len(org_dict))
    return org_dict


def resolve_org_choice(org_dict, user_input):
    """
    Fuzzy-match user input to one organisation.
    Returns (name, anchor_id) or None.
    """
    user_input = (user_input or "").strip()
    if not user_input:
        return None
    if user_input in org_dict:
        return (user_input, org_dict[user_input])
    lower_map = {k.lower(): k for k in org_dict}
    ul = user_input.lower()
    if ul in lower_map:
        k = lower_map[ul]
        return (k, org_dict[k])
    matches = [(k, org_dict[k]) for k in org_dict if ul in k.lower()]
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        logging.warning("Ambiguous match for '%s': %s", user_input, [m[0] for m in matches])
        print(f"  ⚠ '{user_input}' matches multiple orgs — picking first: {matches[0][0]}")
        return matches[0]
    logging.warning("No organisation matched '%s'.", user_input)
    return None


def click_organisation(driver, org_dict, org_name_query):
    """Click the tender-count link for the matched organisation. Returns canonical name or None."""
    match = resolve_org_choice(org_dict, org_name_query)
    if match is None:
        print(f"  ✗ No organisation matched '{org_name_query}'")
        return None
    canonical_name, anchor_id = match
    try:
        el = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.ID, anchor_id))
        )
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
        el.click()
        _wait_for_page_table(driver)
        logging.info("Clicked organisation: %s (id=%s)", canonical_name, anchor_id)
        return canonical_name
    except Exception as e:
        logging.error("Failed to click organisation %s: %s", canonical_name, e)
        return None


def collect_tender_links_current_page(driver):
    """Collect all tender detail links from the current page of the tender listing table."""
    links = []
    try:
        table = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "table"))
        )
    except TimeoutException:
        tables = driver.find_elements(By.TAG_NAME, "table")
        table = None
        for t in tables:
            rows = t.find_elements(By.TAG_NAME, "tr")
            if len(rows) > 2:
                table = t
                break
        if table is None:
            return links

    rows = table.find_elements(By.TAG_NAME, "tr")
    for row in rows:
        cls = row.get_attribute("class") or ""
        if "list_header" in cls:
            continue
        cells = row.find_elements(By.TAG_NAME, "td")
        for cell in cells:
            anchors = cell.find_elements(By.TAG_NAME, "a")
            for a in anchors:
                href = a.get_attribute("href") or ""
                aid = a.get_attribute("id") or ""
                # Tender detail links contain DirectLink and FrontEndViewTender
                if "DirectLink" in aid or "FrontEndViewTender" in href:
                    if href and href not in links:
                        links.append(href)
                    break
    return links


def collect_all_tender_links(driver):
    """
    Iterate through all pages of the tender listing for a department,
    collecting every tender detail URL. Handles NIC pagination (Next link).
    """
    all_links = []
    page_num = 1

    while True:
        page_links = collect_tender_links_current_page(driver)
        all_links.extend(page_links)
        logging.info("Page %d: collected %d tender links (total so far: %d)", page_num, len(page_links), len(all_links))

        # Look for a "Next" pagination link
        next_link = None
        try:
            pagers = driver.find_elements(
                By.XPATH,
                "//a[contains(translate(normalize-space(.), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'next')]"
            )
            for p in pagers:
                if p.is_displayed() and p.is_enabled():
                    next_link = p
                    break
        except Exception:
            pass

        if next_link is None:
            # Also look for ">" style pagination
            try:
                pagers = driver.find_elements(By.XPATH, "//a[normalize-space(.)='>']")
                for p in pagers:
                    if p.is_displayed() and p.is_enabled():
                        next_link = p
                        break
            except Exception:
                pass

        if next_link is None:
            break

        try:
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", next_link)
            next_link.click()
            _wait_for_page_table(driver)
            page_num += 1
        except Exception as e:
            logging.warning("Could not click Next on page %d: %s", page_num, e)
            break

    logging.info("Total tender links collected: %d", len(all_links))
    return all_links


print("Navigation functions defined.")

Navigation functions defined.


## Tender Detail Scraping

In [7]:
TENDER_FIELDS = [
    "Title",
    "Organisation Chain",
    "Tender Value in ₹",
    "Work Description",
    "Location",
    "Bid Submission End Date",
]

TENDER_FIELD_ALIASES = {
    "Tender Value in ₹": [
        "Tender Value in ₹",
        "Tender Value in Rs.",
        "Tender Value",
        "Estimated Cost",
        "Tender Value in Rupees",
    ],
    "Bid Submission End Date": [
        "Bid Submission End Date",
        "Closing Date",
        "Bid Submission Closing Date",
        "Last Date for Submission",
    ],
}

ZIPCODE_LABELS = ["Pin Code", "Pincode", "PIN Code", "ZIP Code", "Zipcode", "Postal Code"]


def _extract_all_fields_from_page(driver):
    """
    Batch-extract all tender fields from the current page in ONE pass.
    Grabs all <td><b>Label</b></td><td>Value</td> pairs at once instead of
    doing separate WebDriverWait calls per field.
    """
    result = {}
    try:
        WebDriverWait(driver, 8).until(
            EC.presence_of_element_located((By.XPATH, "//td/b"))
        )
    except TimeoutException:
        return result

    bold_elements = driver.find_elements(By.XPATH, "//td/b")
    for b in bold_elements:
        try:
            label = b.text.strip()
            if not label:
                continue
            parent_td = b.find_element(By.XPATH, "./ancestor::td")
            value_td = parent_td.find_element(By.XPATH, "./following-sibling::td[1]")
            result[label] = value_td.text.strip()
        except Exception:
            continue
    return result


def _resolve_field(page_data, field_name):
    """Look up a field in pre-extracted page data, trying aliases."""
    val = page_data.get(field_name)
    if val:
        return val
    aliases = TENDER_FIELD_ALIASES.get(field_name, [])
    for alias in aliases:
        val = page_data.get(alias)
        if val:
            return val
    return "Not Found"


def scrape_tender_detail(driver, tender_url):
    """Navigate to a tender detail page and extract all relevant fields (batch mode)."""
    try:
        driver.get(tender_url)
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//td/b"))
        )
    except TimeoutException:
        _recover_page_if_needed(driver, tender_url)
    except Exception as e:
        logging.error("Failed to load tender page %s: %s", tender_url, e)
        return None

    _recover_page_if_needed(driver, tender_url)

    page_data = _extract_all_fields_from_page(driver)
    tender_data = {"Tender Link": tender_url}

    for field in TENDER_FIELDS:
        tender_data[field] = _resolve_field(page_data, field)

    zipcode = "Not Found"
    for label in ZIPCODE_LABELS:
        val = page_data.get(label)
        if val:
            zipcode = val
            break
    tender_data["Zipcode"] = zipcode

    return tender_data


def _recover_page_if_needed(driver, tender_url, max_attempts=3):
    """Handle NIC session/timeout 'Restart' pages."""
    for attempt in range(max_attempts):
        restart = None
        for xpath in [
            "//a[contains(translate(normalize-space(.), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'restart')]",
            "//button[contains(translate(normalize-space(.), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'restart')]",
        ]:
            for el in driver.find_elements(By.XPATH, xpath):
                try:
                    if el.is_displayed() and el.is_enabled():
                        restart = el
                        break
                except Exception:
                    continue
            if restart:
                break

        if restart is None:
            return

        logging.info("Session/timeout — clicking Restart (attempt %d/%d)", attempt + 1, max_attempts)
        try:
            restart.click()
            time.sleep(1)
            driver.get(tender_url)
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, "//td/b"))
            )
        except Exception as e:
            logging.warning("Restart recovery failed: %s", e)


def _scrape_tender_worker(tender_url, org_name):
    """
    Worker function for parallel scraping. Each thread gets its own headless driver.
    Returns tender dict or None.
    """
    driver = init_driver(headless=True)
    try:
        data = scrape_tender_detail(driver, tender_url)
        if data:
            data["Organisation"] = org_name
        return data
    except Exception as e:
        logging.error("Worker error for %s: %s", tender_url, e)
        return None
    finally:
        try:
            driver.quit()
        except Exception:
            pass


def scrape_all_tenders_parallel(tender_links, org_name, max_workers=4):
    """
    Scrape tender detail pages in parallel using multiple headless Chrome instances.
    Each worker gets its own driver — no shared state.
    """
    tenders = []
    total = len(tender_links)
    completed = 0
    lock = threading.Lock()

    print(f"  Scraping {total} tenders with {max_workers} parallel workers...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_url = {
            executor.submit(_scrape_tender_worker, url, org_name): url
            for url in tender_links
        }

        for future in as_completed(future_to_url):
            url = future_to_url[future]
            with lock:
                completed += 1
            try:
                data = future.result()
                if data:
                    tenders.append(data)
                    pct = round(completed / total * 100, 1)
                    logging.info("[%d/%d] %.1f%% — %s", completed, total, pct, (data.get("Title") or "")[:80])
                else:
                    logging.warning("[%d/%d] Failed: %s", completed, total, url)
            except Exception as e:
                logging.error("[%d/%d] Exception: %s — %s", completed, total, url, e)

            if completed % 20 == 0 or completed == total:
                print(f"  [{completed}/{total}] {round(completed/total*100, 1)}% done")

    return tenders


def filter_tenders_by_value(tenders, min_r, max_r):
    """Keep tenders whose value falls within [min_r, max_r] rupees."""
    kept, dropped_missing, dropped_range = [], 0, 0
    for t in tenders:
        raw = t.get("Tender Value in ₹", "")
        val = parse_inr_amount_to_rupees(raw)
        if val is None:
            dropped_missing += 1
            continue
        if val < min_r or val > max_r:
            dropped_range += 1
            continue
        kept.append(t)
    logging.info("Value filter: kept %d, dropped %d (missing), %d (out of range)", len(kept), dropped_missing, dropped_range)
    print(f"  Value filter: {len(kept)} kept, {dropped_missing} missing value, {dropped_range} out of range")
    return kept


print("Tender scraping functions defined.")

Tender scraping functions defined.


## BOQ Download — ZIP download with CAPTCHA handling, extraction, PDF conversion

In [8]:
def find_download_zip_link(driver):
    """Locate the 'Download as zip file' link on a tender detail page."""
    xpaths = [
        "//a[contains(translate(normalize-space(string(.)), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'download a zip')]",
        "//a[contains(translate(normalize-space(string(.)), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'zip file')]",
        "//a[contains(translate(normalize-space(string(.)), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'download') and contains(translate(normalize-space(string(.)), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'zip')]",
    ]
    for xp in xpaths:
        for a in driver.find_elements(By.XPATH, xp):
            try:
                if a.is_displayed() and a.is_enabled():
                    return a
            except Exception:
                continue
    for a in driver.find_elements(By.TAG_NAME, "a"):
        try:
            txt = (a.text or "").lower()
            if "zip" in txt and "download" in txt and a.is_displayed():
                return a
        except Exception:
            continue
    return None


def _safe_click(driver, element):
    try:
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", element)
        time.sleep(0.3)
        element.click()
        return True
    except Exception as e:
        logging.warning("Click failed: %s", e)
        return False


def wait_for_zip_download(download_dir, before_names, timeout=180):
    """Wait until a new .zip file appears and Chrome .crdownload temps are gone."""
    deadline = time.time() + timeout
    before_names = set(before_names)
    while time.time() < deadline:
        try:
            names = os.listdir(download_dir)
        except OSError:
            time.sleep(0.5)
            continue
        if any(n.endswith(".crdownload") for n in names):
            time.sleep(0.5)
            continue
        new_zips = [n for n in names if n.endswith(".zip") and n not in before_names]
        if new_zips:
            new_zips.sort(
                key=lambda n: os.path.getmtime(os.path.join(download_dir, n)),
                reverse=True,
            )
            return os.path.join(download_dir, new_zips[0])
        time.sleep(0.5)
    return None


def try_download_zip(driver, download_dir, tender_label, tender_url=None):
    """
    Click 'Download as zip'. If ZIP appears quickly → no CAPTCHA.
    Otherwise prompt the user in the notebook to solve CAPTCHA and press Enter.
    """
    link = find_download_zip_link(driver)
    if not link:
        logging.warning("No zip download link for: %s", tender_label)
        return None

    before = set(os.listdir(download_dir))
    if not _safe_click(driver, link):
        return None

    # Quick check — some tenders download without CAPTCHA
    path = wait_for_zip_download(download_dir, before, timeout=25)
    if path:
        logging.info("ZIP downloaded (no CAPTCHA) for: %s", tender_label)
        return path

    # CAPTCHA required — prompt user
    for attempt in range(1, 4):
        print(
            f"\n  >>> CAPTCHA needed ({attempt}/3) — {tender_label}\n"
            "      Solve the CAPTCHA in the Chrome window, then press Enter here.\n"
        )
        try:
            input("  Press Enter when ready: ")
        except EOFError:
            logging.error("EOF on input; aborting download for %s", tender_label)
            return None

        # Recover if we landed on a timeout/restart page
        _recover_page_if_needed(driver, tender_url or "")

        link2 = find_download_zip_link(driver)
        if not link2:
            logging.warning("No zip link after CAPTCHA wait (round %d): %s", attempt, tender_label)
            if tender_url:
                driver.get(tender_url)
                time.sleep(2)
            continue

        before_round = set(os.listdir(download_dir))
        if not _safe_click(driver, link2):
            continue

        path = wait_for_zip_download(download_dir, before_round, timeout=180)
        if path:
            logging.info("ZIP downloaded after CAPTCHA for: %s", tender_label)
            return path
        logging.warning("No ZIP after click (round %d): %s", attempt, tender_label)

    logging.warning("Gave up on ZIP download for: %s", tender_label)
    return None


def extract_boq_from_zip(zip_path, boq_dir, tender_title, seq_index=0):
    """
    Find BOQ spreadsheet inside ZIP, convert to PDF, save in boq_dir.
    Returns (pdf_path, status_str).
    """
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            members = [n for n in zf.namelist() if not n.endswith("/")]
            if not members:
                return None, "empty_zip"

            base_names = [(n, os.path.basename(n)) for n in members]
            boq_files = [n for n, b in base_names if "boq" in b.lower()]
            candidates = boq_files or [
                n for n, b in base_names
                if b.lower().endswith((".xlsx", ".xls", ".csv"))
            ]
            if not candidates:
                return None, "no_boq_in_zip"

            inner = candidates[0]
            ext = os.path.splitext(inner)[1].lower()
            raw_bytes = zf.read(inner)
    except (zipfile.BadZipFile, Exception) as e:
        logging.error("Bad zip %s: %s", zip_path, e)
        return None, f"bad_zip: {e}"

    # Parse the spreadsheet into a DataFrame
    bio = io.BytesIO(raw_bytes)
    try:
        if ext == ".csv":
            df = pd.read_csv(bio, header=None, encoding_errors="replace")
        elif ext == ".xlsx":
            xls = pd.ExcelFile(bio, engine="openpyxl")
            parts = []
            for sheet in xls.sheet_names:
                parts.append(pd.read_excel(xls, sheet_name=sheet, header=None))
            df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
        elif ext == ".xls":
            xls = pd.ExcelFile(bio, engine="xlrd")
            parts = []
            for sheet in xls.sheet_names:
                parts.append(pd.read_excel(xls, sheet_name=sheet, header=None))
            df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
        else:
            xls = pd.ExcelFile(bio)
            parts = []
            for sheet in xls.sheet_names:
                parts.append(pd.read_excel(xls, sheet_name=sheet, header=None))
            df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    except Exception as e:
        logging.error("Could not read BOQ spreadsheet from %s: %s", zip_path, e)
        return None, f"read_error: {e}"

    if df.empty:
        return None, "empty_boq"

    # Write BOQ to PDF
    safe_title = sanitize_folder_name(tender_title or f"tender_{seq_index:04d}")
    if len(safe_title) > 120:
        safe_title = safe_title[:120].rstrip("_")
    pdf_path = unique_path_in_dir(boq_dir, safe_title, ext=".pdf")

    try:
        _write_boq_dataframe_to_pdf(df, pdf_path, tender_title or safe_title)
        logging.info("BOQ PDF: %s", pdf_path)
        return pdf_path, "ok"
    except Exception as e:
        logging.error("PDF write failed for %s: %s", tender_title, e)
        return None, f"pdf_error: {e}"


def _write_boq_dataframe_to_pdf(df, pdf_path, title):
    """Render a raw DataFrame as a simple multi-page PDF table."""
    from reportlab.lib import colors
    from reportlab.lib.enums import TA_LEFT
    from reportlab.lib.pagesizes import A4, landscape
    from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
    from reportlab.lib.units import mm
    from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle
    import html as html_module

    LM = RM = 14 * mm
    TM = BM = 12 * mm
    page_w, page_h = landscape(A4)
    usable_w = page_w - LM - RM

    doc = SimpleDocTemplate(
        pdf_path, pagesize=landscape(A4),
        leftMargin=LM, rightMargin=RM, topMargin=TM, bottomMargin=BM,
    )
    styles = getSampleStyleSheet()
    title_style = ParagraphStyle(
        "BoqTitle", parent=styles["Title"], fontSize=13, leading=16,
        textColor=colors.HexColor("#1F4E79"), spaceAfter=8,
    )
    cell_style = ParagraphStyle(
        "BoqCell", fontName="Helvetica", fontSize=7, leading=9,
        leftIndent=2, rightIndent=2, spaceBefore=1, spaceAfter=1, alignment=TA_LEFT,
    )
    header_style = ParagraphStyle(
        "BoqHead", parent=cell_style, fontName="Helvetica-Bold",
        fontSize=7, leading=9, textColor=colors.whitesmoke,
    )

    def safe_para(val, style):
        s = "" if pd.isna(val) else str(val).strip()
        s = html_module.escape(s).replace("\n", "<br/>")
        return Paragraph(s or " ", style)

    flow = [Paragraph(html_module.escape(title)[:200], title_style), Spacer(1, 4)]

    ncols = df.shape[1]
    col_w = usable_w / max(ncols, 1)
    col_widths = [col_w] * ncols

    # Use first non-blank row as header if it looks like one, otherwise just Col_0..Col_n
    header_labels = [f"Col {j}" for j in range(ncols)]
    for idx in range(min(5, len(df))):
        row_vals = [str(v).strip() if pd.notna(v) else "" for v in df.iloc[idx]]
        non_empty = sum(1 for v in row_vals if v)
        if non_empty >= ncols * 0.5:
            header_labels = row_vals
            df = df.iloc[idx + 1:].reset_index(drop=True)
            break

    header_row = [safe_para(h, header_style) for h in header_labels]
    table_data = [header_row]
    for _, row in df.iterrows():
        table_data.append([safe_para(row.iloc[j], cell_style) for j in range(ncols)])

    t = Table(table_data, colWidths=col_widths, repeatRows=1, splitByRow=1)
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1F4E79")),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("GRID", (0, 0), (-1, -1), 0.3, colors.HexColor("#B0B0B0")),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#F0F5FA")]),
        ("TOPPADDING", (0, 0), (-1, -1), 4),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
        ("LEFTPADDING", (0, 0), (-1, -1), 3),
        ("RIGHTPADDING", (0, 0), (-1, -1), 3),
    ]))
    flow.append(t)
    doc.build(flow)


def download_boqs_for_tenders(tenders, boq_dir, staging_dir, driver):
    """
    Download ZIP and extract BOQ PDF for each tender.
    Mutates tender dicts in-place with BOQ_Status and BOQ_PDF fields.
    """
    os.makedirs(boq_dir, exist_ok=True)
    total = len(tenders)

    for i, t in enumerate(tenders, start=1):
        t.setdefault("BOQ_Status", "")
        t.setdefault("BOQ_PDF", "")
        link = t.get("Tender Link")
        if not link:
            t["BOQ_Status"] = "no_link"
            continue

        label = f"{i}/{total} — {(t.get('Title') or '')[:60]}"
        try:
            driver.get(link)
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, "//td/b"))
            )
        except Exception as e:
            logging.error("Failed to open tender %s: %s", label, e)
            t["BOQ_Status"] = "page_error"
            continue

        _recover_page_if_needed(driver, link)

        zip_path = try_download_zip(driver, staging_dir, label, tender_url=link)
        if not zip_path:
            t["BOQ_Status"] = "download_failed"
            continue

        # Move ZIP into BOQs folder with a meaningful name
        proj_base = sanitize_project_zip_basename(t, i)
        dest_zip = unique_path_in_dir(boq_dir, proj_base, ext=".zip")
        try:
            if os.path.abspath(zip_path) != os.path.abspath(dest_zip):
                shutil.move(zip_path, dest_zip)
        except OSError as e:
            logging.warning("Could not move zip: %s", e)
            dest_zip = zip_path

        pdf_path, status = extract_boq_from_zip(dest_zip, boq_dir, t.get("Title", ""), seq_index=i)
        t["BOQ_Status"] = status
        t["BOQ_PDF"] = pdf_path or ""

        if i % 5 == 0 or i == total:
            print(f"  BOQ [{i}/{total}] — status: {status}")


print("BOQ download functions defined.")

BOQ download functions defined.


## Excel Export — styled output matching the column layout: Sno, Title, Organisation Chain, Closing Date, Amount, Work Description, Location, Zipcode, District, State/U.T.

In [9]:
def build_export_dataframe(tenders):
    """
    Build a DataFrame with the standard column layout from scraped tender dicts.
    Columns: Title, Tender Link, Organisation Chain, Closing Date, Amount,
             Work Description, Location, Zipcode, District, State/U.T.
    """
    df = pd.DataFrame(tenders)

    rename_map = {
        "Title": "Title",
        "Organisation Chain": "Organisation Chain",
        "Bid Submission End Date": "Closing Date",
        "Tender Value in ₹": "Amount",
        "Work Description": "Work Description",
        "Location": "Location",
    }
    df = df.rename(columns=rename_map)

    required = ["Title", "Tender Link", "Organisation Chain", "Closing Date",
                 "Amount", "Work Description", "Location", "Zipcode"]
    for col in required:
        if col not in df.columns:
            df[col] = ""

    # District and State from the Location / Zipcode fields if possible
    if "District" not in df.columns:
        df["District"] = ""
    if "State/U.T." not in df.columns:
        df["State/U.T."] = ""

    # Try to extract state/district from Organisation Chain
    # (NIC format: "Ministry||Dept||State||Sub-org")
    for idx, row in df.iterrows():
        chain = str(row.get("Organisation Chain", ""))
        parts = [p.strip() for p in chain.replace("||", "|").split("|") if p.strip()]
        if not row.get("State/U.T.") and len(parts) >= 3:
            df.at[idx, "State/U.T."] = parts[-1] if len(parts[-1]) < 40 else ""
        if not row.get("District") and len(parts) >= 4:
            df.at[idx, "District"] = parts[-2] if len(parts[-2]) < 40 else ""

    return df[required + ["District", "State/U.T."]]


def export_formatted_excel(df, path, sheet_name="Tenders"):
    """Write styled Excel with header row, hyperlinked titles, zebra rows, borders."""
    cols_order = [
        "Title", "Organisation Chain", "Closing Date", "Amount",
        "Work Description", "Location", "Zipcode", "District", "State/U.T.",
    ]
    for c in cols_order:
        if c not in df.columns:
            df[c] = ""

    titles = [("" if pd.isna(t) else str(t).strip()) for t in df["Title"]]
    links = [("" if pd.isna(u) else str(u).strip()) for u in df["Tender Link"]]
    write_df = df[cols_order].copy()

    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        write_df.to_excel(writer, sheet_name=sheet_name, index=True, index_label="Sno")

    wb = load_workbook(path)
    ws = wb[sheet_name]

    thin = Side(style="thin", color="B4B4B4")
    grid_border = Border(left=thin, right=thin, top=thin, bottom=thin)
    header_fill = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
    header_font = Font(bold=True, color="FFFFFF", size=11)
    fill_alt = PatternFill(start_color="E8F0F8", end_color="E8F0F8", fill_type="solid")
    fill_base = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")
    link_font = Font(color="0563C1", underline="single", size=11)
    body_font = Font(size=11)

    max_col = ws.max_column
    title_col_idx = 2  # col A = Sno, col B = Title

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = grid_border

    for r in range(2, ws.max_row + 1):
        row_fill = fill_alt if (r - 2) % 2 == 0 else fill_base
        for c in range(1, max_col + 1):
            cell = ws.cell(row=r, column=c)
            cell.border = grid_border
            cell.fill = row_fill
            if c == title_col_idx:
                url = links[r - 2] if r - 2 < len(links) else ""
                tit = titles[r - 2] if r - 2 < len(titles) else ""
                if url.startswith("http"):
                    cell.hyperlink = url
                    cell.value = tit if tit and tit != "nan" else url
                    cell.font = link_font
                else:
                    cell.value = tit
                    cell.font = body_font
                cell.alignment = Alignment(vertical="top", wrap_text=True)
            else:
                cell.font = body_font
                cell.alignment = Alignment(vertical="top", wrap_text=True)

    col_widths = {1: 7, 2: 40, 3: 32, 4: 18, 5: 16, 6: 40, 7: 18, 8: 10, 9: 18, 10: 14}
    for col_idx, w in col_widths.items():
        if col_idx <= max_col:
            ws.column_dimensions[get_column_letter(col_idx)].width = w

    ws.freeze_panes = "A2"
    wb.save(path)
    logging.info("Excel saved: %s", path)


print("Excel export functions defined.")

Excel export functions defined.


## Main Execution

Run this cell to start scraping. Everything runs **headless** (no visible browser).
- **Phase 1**: Discovers organisations and collects tender links (single headless browser)
- **Phase 2**: Scrapes tender details in **parallel** using multiple headless workers
- **BOQ downloads**: Chrome opens visibly **only** for BOQ ZIP downloads (CAPTCHA may need manual solving)

In [10]:
# ═══════════════════════════════════════════════════════════════════
#  PHASE 1 — Discover organisations and collect tender links
#            (single headless browser — fast, invisible)
# ═══════════════════════════════════════════════════════════════════

print("=" * 60)
print("PHASE 1: Discovering organisations and collecting tender URLs")
print("         (running headless — no browser window)")
print("=" * 60)

driver = init_driver(headless=True)
navigate_to_org_listing(driver)
org_dict = get_all_organisations(driver)

if not org_dict:
    print("ERROR: No organisations found on the page. Check the portal or your network.")
    driver.quit()
    raise RuntimeError("No organisations found.")

print(f"\nFound {len(org_dict)} organisations on the portal.")

org_jobs = []
for query in ORGANISATIONS:
    match = resolve_org_choice(org_dict, query)
    if match:
        org_jobs.append(match)
        print(f"  OK: '{query}' -> {match[0]}")
    else:
        print(f"  MISS: '{query}' — no match found")

if not org_jobs:
    print("\nNo matching organisations. Available organisations:")
    for idx, name in enumerate(sorted(org_dict.keys()), 1):
        print(f"  {idx}. {name}")
    driver.quit()
    raise RuntimeError("No organisations matched. Update ORGANISATIONS list above and re-run.")

print(f"\nWill scrape {len(org_jobs)} organisation(s): {', '.join(n for n, _ in org_jobs)}")

# ═══════════════════════════════════════════════════════════════════
#  Collect tender links for ALL orgs using the same headless driver
# ═══════════════════════════════════════════════════════════════════

org_tender_links = {}
for org_name, org_anchor_id in org_jobs:
    navigate_to_org_listing(driver)
    org_dict_fresh = get_all_organisations(driver)
    canonical = click_organisation(driver, org_dict_fresh, org_name)
    if canonical is None:
        print(f"  Could not click organisation '{org_name}' — skipping.")
        org_tender_links[org_name] = []
        continue
    links = collect_all_tender_links(driver)
    org_tender_links[org_name] = links
    print(f"  {org_name}: {len(links)} tender link(s) found")

driver.quit()
print("\nPhase 1 complete — headless browser closed.")

# ═══════════════════════════════════════════════════════════════════
#  PHASE 2 — Parallel scraping of tender details per organisation
#            (N headless workers, no visible browser)
# ═══════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print(f"PHASE 2: Scraping tender details ({NUM_WORKERS} parallel workers, headless)")
print("=" * 60)

summary_results = []
boq_driver = None
boq_staging_dir = os.path.join(output_root, "_chrome_staging")

try:
    for org_name, _ in org_jobs:
        tender_links = org_tender_links.get(org_name, [])
        print(f"\n{'─' * 50}")
        print(f"Organisation: {org_name} ({len(tender_links)} tenders)")
        print(f"{'─' * 50}")

        if not tender_links:
            summary_results.append({"org": org_name, "total": 0, "filtered": 0, "boqs": 0})
            continue

        safe_name = sanitize_folder_name(org_name)
        org_dir = os.path.join(output_root, safe_name)
        boq_dir = os.path.join(org_dir, "BOQs")
        os.makedirs(boq_dir, exist_ok=True)
        excel_path = os.path.join(org_dir, f"{safe_name}_tenders.xlsx")

        tenders = scrape_all_tenders_parallel(tender_links, org_name, max_workers=NUM_WORKERS)
        print(f"  Scraped {len(tenders)} tender(s)")

        tenders_filtered = filter_tenders_by_value(tenders, min_rupees, max_rupees)

        if not tenders_filtered:
            print(f"  No tenders in the specified value range — no Excel/BOQ output.")
            summary_results.append({"org": org_name, "total": len(tenders), "filtered": 0, "boqs": 0})
            continue

        df = build_export_dataframe(tenders_filtered)
        export_formatted_excel(df, excel_path)
        print(f"  Excel saved: {excel_path}")

        boq_count = 0
        if DOWNLOAD_BOQS:
            if boq_driver is None:
                os.makedirs(boq_staging_dir, exist_ok=True)
                boq_driver = init_driver(headless=False, download_dir=boq_staging_dir)
                print(f"\n  Chrome opened for BOQ downloads (CAPTCHA may require interaction).")
                print(f"  Staging: {boq_staging_dir}")

            print(f"  Downloading BOQs into: {boq_dir}")
            download_boqs_for_tenders(tenders_filtered, boq_dir, boq_staging_dir, boq_driver)
            boq_count = sum(1 for t in tenders_filtered if t.get("BOQ_Status") == "ok")
            print(f"  BOQs downloaded: {boq_count}/{len(tenders_filtered)}")

        summary_results.append({
            "org": org_name,
            "total": len(tenders),
            "filtered": len(tenders_filtered),
            "boqs": boq_count,
        })

finally:
    if boq_driver is not None:
        try:
            boq_driver.quit()
        except Exception:
            pass

print("\n" + "=" * 60)
print("Scraping complete!")
print("=" * 60)

2026-04-11 13:09:08,533 - INFO - ====== WebDriver manager ======


PHASE 1: Discovering organisations and collecting tender URLs


2026-04-11 13:09:09,019 - INFO - Get LATEST chromedriver version for google-chrome
2026-04-11 13:09:09,247 - INFO - Get LATEST chromedriver version for google-chrome
2026-04-11 13:09:09,490 - INFO - Driver [/Users/vakul/.wdm/drivers/chromedriver/mac64/146.0.7680.165/chromedriver-mac-arm64/chromedriver] found in cache
2026-04-11 13:09:36,521 - INFO - Navigated to Tender by Organisation page via direct URL.
2026-04-11 13:09:39,184 - INFO - Found 81 organisations on the listing page.



Found 81 organisations on the portal.
  ✓ 'National Highways Authority of India' → National Highways Authority of India

Will scrape 1 organisation(s): National Highways Authority of India

PHASE 2: Scraping tender details per organisation

──────────────────────────────────────────────────
Organisation: National Highways Authority of India
──────────────────────────────────────────────────


2026-04-11 13:10:00,960 - INFO - Navigated to Tender by Organisation page via direct URL.
2026-04-11 13:10:03,510 - INFO - Found 81 organisations on the listing page.
2026-04-11 13:10:09,069 - INFO - Clicked organisation: National Highways Authority of India (id=DirectLink_56)
2026-04-11 13:10:23,212 - INFO - Page 1: collected 309 tender links (total so far: 309)
2026-04-11 13:10:23,234 - INFO - Total tender links collected: 309


...


  >>> CAPTCHA needed (1/3) — 49/53 — Construction of 3 lane ROB on LHS at Km.1034.092 (existing r
      Solve the CAPTCHA in the Chrome window, then press Enter here.



2026-04-11 13:36:57,074 - INFO - ZIP downloaded after CAPTCHA for: 49/53 — Construction of 3 lane ROB on LHS at Km.1034.092 (existing r
2026-04-11 13:37:05,443 - INFO - ZIP downloaded (no CAPTCHA) for: 50/53 — Construction of PQC service road, Construction of RCC Drain 


  BOQ [50/53] — status: no_boq_in_zip


2026-04-11 13:37:10,315 - INFO - ZIP downloaded (no CAPTCHA) for: 51/53 — Construction of PQC service road and RCC Drains and other Mi
2026-04-11 13:37:15,076 - INFO - ZIP downloaded (no CAPTCHA) for: 52/53 — Construction of 1 Nos VUP and 1 Nos 2 Lane Grade Separator i
2026-04-11 13:37:40,384 - INFO - ZIP downloaded (no CAPTCHA) for: 53/53 — Rehabilitation and upgradation of Ramachandrapuram Village l


  BOQ [53/53] — status: no_boq_in_zip
  BOQs downloaded: 7/53

Scraping complete!


## Summary

In [11]:
print("=" * 60)
print("  SCRAPING SUMMARY")
print("=" * 60)
print(f"  Output root : {output_root}")
print(f"  Value filter: ₹{MIN_CRORE} Cr – ₹{MAX_CRORE} Cr")
print(f"  Log file    : {log_file}")
print()

if summary_results:
    summary_df = pd.DataFrame(summary_results)
    summary_df.columns = ["Organisation", "Total Tenders", "After Filter", "BOQs Downloaded"]
    print(summary_df.to_string(index=False))
else:
    print("  No results.")

print()
print("  Folder structure:")
for root, dirs, files in os.walk(output_root):
    level = root.replace(output_root, "").count(os.sep)
    indent = "  " * (level + 1)
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 2)
    for f in files[:20]:
        print(f"{sub_indent}{f}")
    if len(files) > 20:
        print(f"{sub_indent}... and {len(files) - 20} more files")

print("\nDone!")

  SCRAPING SUMMARY
  Output root : /Users/vakul/Documents/Projects & Ideas/TendersScrapper-RPG/RPGOYAL EPROC Automation/national_Projects_dep_2026-04-11_13-08-49
  Value filter: ₹8 Cr – ₹50 Cr
  Log file    : /Users/vakul/Documents/Projects & Ideas/TendersScrapper-RPG/RPGOYAL EPROC Automation/logs/national_scraper_2026-04-11_13-08-09.log

                        Organisation  Total Tenders  After Filter  BOQs Downloaded
National Highways Authority of India            309            53                7

  Folder structure:
  national_Projects_dep_2026-04-11_13-08-49/
    .DS_Store
    _chrome_staging/
      work_290875.zip
    National Highways Authority of India/
      National Highways Authority of India_tenders.xlsx
      .DS_Store
      ~$National Highways Authority of India_tenders.xlsx
      BOQs/
        Engagement of user fee collection agency on the basis of CB through etender at Dharmaram FP in the State of Telangana and upkeep_maintenance of adjace.zip
        Construction of